In [1]:
import pandas as pd
import gc
import numpy as np

In [2]:
df3 = pd.read_pickle('../data/df3.pkl')

In [ ]:
df3.info()
df3.shape # (3846784, 30)
df3.head(20)

### Information on the Patron Categories: 

- Student (Primary 1 to JC/Poly) --> 7 - 19 years old
- Adult --> 20 - 59 years old
- Senior Citizen --> 60 years and above

Average Walking Speeds by Age:
- Children (6-12 years): Around 3.0 to 4.5 km/h (1.9 to 2.8 mph)
- Adolescents (13-19 years): Around 4.0 to 5.0 km/h (2.5 to 3.1 mph)
- Adults (20-59 years): Around 5.0 to 6.0 km/h (3.1 to 3.7 mph)
- Older Adults (60+ years): Around 4.0 to 5.0 km/h (2.5 to 3.1 mph)

To calculate the walking speed of each group, I take the midpoint of the range. 
For Children and Adolescents, I took the midpoint of both ranges and took the average.

In [3]:
# We first filter for 3 relevant PATRON_CATG_ID_NUM.
patron_map = {1: 'Adult', 3: 'Student', 4: 'Senior Citizen'}
df3['PATRON_CATG_DESC_TXT'] = df3['PATRON_CATG_ID_NUM'].map(patron_map)

walking_speed_ms = {
    'Student': 4.125 / 3.6,      # (3.75 + 4.5) / 2 = 4.125 km/h → ~1.146 m/s
    'Adult': 5.5 / 3.6,          # ~1.528 m/s
    'Senior Citizen': 4.5 / 3.6  # 1.25 m/s
}

df3['walking_speed_ms'] = df3['PATRON_CATG_DESC_TXT'].map(walking_speed_ms)

In [ ]:
df3.head(20)

# Rule Based Classification

#### 1. Binary Criteria
- Drop rows with missing critical data
- The journey stage has no subsequent recorded trips on the same day and is therefore treated as a journey termination. It is the last ride of the day for the commuter.
    - Implementation: df3 is already sorted by CRD_NUM and ENTRY_TM. Flag last row of each CRD_NUM
- 2 consecutive journey stages on the same bus service or train station indicates a return trip or intermediate activity. This should be classified as a new journey.
    - Implementation: Shift BUS_SVC_NUM and ORIG_LOC_ID_NUM down by 1 row to get the next stage's values, then flag rows where all 3 conditions are true simultaneously: same commuter, same bus service, and the alighting stop of stage 1 equals the boarding stop of stage 2.


In [4]:
df3 = df3.sort_values(["CRD_NUM", "ENTRY_TM"]).reset_index(drop=True)

In [5]:
df3["next_ENTRY_TM"] = df3.groupby("CRD_NUM")["ENTRY_TM"].shift(-1)
df3["next_ORIG_LOC_ID_NUM"] = df3.groupby("CRD_NUM")["ORIG_LOC_ID_NUM"].shift(-1)
df3["next_BUS_SVC_NUM"] = df3.groupby("CRD_NUM")["BUS_SVC_NUM"].shift(-1)
df3["next_TRNSPT_MODE_CD"] = df3.groupby("CRD_NUM")["TRNSPT_MODE_CD"].shift(-1)

In [ ]:
df3.head(20)

In [6]:
df3["is_last_stage"] = df3["next_ENTRY_TM"].isna()

df3['missing_info'] = (
    df3['ENTRY_TM'].isna() |
    df3['EXIT_TM'].isna() |
    df3['ORIG_LATITUDE'].isna() |
    df3['ORIG_LONGITUDE'].isna() |
    df3['DEST_LATITUDE'].isna() |
    df3['DEST_LONGITUDE'].isna() |
    df3["ORIG_LOC_ID_NUM"].isna() |
    df3["DEST_LOC_ID_NUM"].isna()
)

In [ ]:
df3.head(20)

In [7]:
#create next bus and next station columns by shifting
df3["same_bus_service"] = (
    df3["BUS_SVC_NUM"].notna() &
    df3["next_BUS_SVC_NUM"].notna() &
    (df3["BUS_SVC_NUM"] == df3["next_BUS_SVC_NUM"])
)
df3["same_station_consecutive"] = (
    df3["DEST_STATION_NAME"].notna() &
    df3["next_orig_station"].notna() &
    (df3["DEST_STATION_NAME"] == df3["next_orig_station"])
)

In [ ]:
df3.head(20)

In [8]:
df3["return_or_intermediate"] = (
    df3["same_bus_service"] |
    df3["same_station_consecutive"]
)

In [9]:
df3["journey_termination_flag"] = (
    df3["is_last_stage"] |
    df3["missing_info"] |
    df3["return_or_intermediate"]
)

In [ ]:
df3['journey_termination_flag'].value_counts()


In [ ]:
df3[df3['journey_termination_flag']].head(40)

In [ ]:
df3.head(20)

In [10]:
# Summary counts
print("Unique commuters:", df3["CRD_NUM"].nunique())
print("Last stages:", df3["is_last_stage"].sum())

print("\nMissing info:")
print(df3["missing_info"].value_counts(dropna=False))

print("\nSame bus service:")
print(df3["same_bus_service"].value_counts(dropna=False))

print("\nSame station consecutive:")
print(df3["same_station_consecutive"].value_counts(dropna=False))

print("\nReturn / intermediate:")
print(df3["return_or_intermediate"].value_counts(dropna=False))

print("\nJourney termination flag:")
print(df3["journey_termination_flag"].value_counts(dropna=False))

Unique commuters: 1212990
Last stages: 1212990

Missing info:
missing_info
False    3817059
True       29725
Name: count, dtype: int64

Same bus service:
same_bus_service
False    3521176
True      325608
Name: count, dtype: int64

Same station consecutive:
same_station_consecutive
False    3068298
True      778486
Name: count, dtype: int64

Return / intermediate:
return_or_intermediate
False    2795500
True     1051284
Name: count, dtype: int64

Journey termination flag:
journey_termination_flag
True     2281428
False    1565356
Name: count, dtype: int64


In [ ]:
# If you want to see flag rows
display(
    df3.loc[
        df3["journey_termination_flag"],
        [
            "CRD_NUM",
            "ENTRY_TM",
            "EXIT_TM",
            "ORIG_STATION_NAME",
            "DEST_STATION_NAME",
            "next_orig_station",
            "BUS_SVC_NUM",
            "next_BUS_SVC_NUM",
            "is_last_stage",
            "missing_info",
            "same_bus_service",
            "same_station_consecutive",
            "return_or_intermediate",
            "journey_termination_flag"
        ]
    ].head(20)
)

In [ ]:
# If you want to see transfers
display(
    df3.loc[
        ~df3["journey_termination_flag"],
        [
            "CRD_NUM",
            "ENTRY_TM",
            "EXIT_TM",
            "ORIG_STATION_NAME",
            "DEST_STATION_NAME",
            "next_orig_station",
            "BUS_SVC_NUM",
            "next_BUS_SVC_NUM",
            "is_last_stage",
            "missing_info",
            "same_bus_service",
            "same_station_consecutive",
            "return_or_intermediate",
            "journey_termination_flag"
        ]
    ].head(20)
)

In [12]:
# Reasons why flagged.
df3["journey_termination_reason"] = np.select(
    [
        df3["is_last_stage"],
        df3["missing_info"],
        df3["return_or_intermediate"]
    ],
    [
        "last_stage",
        "missing_info",
        "return_or_intermediate"
    ],
    default="candidate_transfer"
)

In [13]:
print(df3["journey_termination_reason"].value_counts(dropna=False))

journey_termination_reason
candidate_transfer        1565356
last_stage                1212990
return_or_intermediate    1047113
missing_info                21325
Name: count, dtype: int64


In [ ]:
#Don't need I think.. pls review

'''#start of previous code
# Drop rows with missing critical data
critical_cols = [
    'ENTRY_TM', 'EXIT_TM',
    'ORIG_LOC_ID_NUM', 'DEST_LOC_ID_NUM',
    'ORIG_LATITUDE', 'ORIG_LONGITUDE',
    'DEST_LATITUDE', 'DEST_LONGITUDE'
]

before = len(df3)
df3 = df3.dropna(subset=critical_cols).reset_index(drop=True)
after = len(df3)

print(f"Rows dropped: {before - after:,}")
print(f"Rows remaining: {after:,}")'''

In [ ]:
#Already have better checks from before. 

'''# Binary criteria

# Shift to get next row's CRD_NUM
df3['next_CRD_NUM'] = df3['CRD_NUM'].shift(-1)

# Last stage = next row belongs to a different commuter
df3['is_last_stage'] = df3['CRD_NUM'] != df3['next_CRD_NUM'] # This tells us the last ride for the specific commuter

print(df3['is_last_stage'].value_counts())
print(f"\nUnique commuters: {df3['CRD_NUM'].nunique():,}") # Check that no. if True in is_last_stage == No. of unique commuters'''

In [ ]:
# Repeated
'''# Shift needed columns
df3['next_ORIG_LOC_ID_NUM'] = df3['ORIG_LOC_ID_NUM'].shift(-1)
df3['next_BUS_SVC_NUM']     = df3['BUS_SVC_NUM'].shift(-1)

# Flag same service return
df3['same_service_return'] = (
    (df3['CRD_NUM'] == df3['next_CRD_NUM']) &
    (df3['BUS_SVC_NUM'] == df3['next_BUS_SVC_NUM']) &
    (df3['DEST_LOC_ID_NUM'] == df3['next_ORIG_LOC_ID_NUM'])
)

print(df3['same_service_return'].value_counts())'''

In [ ]:
# Not sure
'''print("=== LAST TRIP COMPARISON ===")
print("Yours (is_last_trip):")
print(df3['is_last_trip'].value_counts(), "\n")

print("Hers (is_last_stage):")
print(df3['is_last_stage'].value_counts(), "\n")

print("Difference in TRUE count:",
      df3['is_last_trip'].sum() - df3['is_last_stage'].sum())'''

In [ ]:
# Not sure
'''print("\n=== RETURN TRIP COMPARISON ===")
print("Yours (return_or_intermediate):")
print(df3['return_or_intermediate'].value_counts(), "\n")

print("Hers (same_service_return):")
print(df3['same_service_return'].value_counts(), "\n")

print("Difference in TRUE count:",
      df3['return_or_intermediate'].sum() - df3['same_service_return'].sum())'''

In [ ]:
# Not sure
'''print("\n=== AGREEMENT ===")

last_agree = (df3['is_last_trip'] == df3['is_last_stage']).mean()
return_agree = (df3['return_or_intermediate'] == df3['same_service_return']).mean()

print(f"Last trip agreement: {last_agree:.4f}")
print(f"Return trip agreement: {return_agree:.4f}")'''

#### 2. Temporal criteria: 
- The binary criteria gives us pairs of consecutive stages that could be transfers. 
- Temporal criteria finds the time gap between alighting from stage 1 and boarding stage 2
- Implementation: 
    - gap = ENTRY_TM of stage 2 - EXIT_TM of stage 1. 
    - Shift ENTRY_TM and TRNSPT_MODE_CD down by 1 row to get the next stage's values. Compute the time gap between current EXIT_TM and next ENTRY_TM. Assign threshold (15 or 45 mins) based on transfer type, then flag if gap exceeds it.
    - Flag if gap > allowance.
    - Allowance = Walking speed * walking distance



* Note this is how TRNSPRT_MODE_CD is mapped.
1    BUS
2    RTS
3    BUS-RTS (interchange)

In [14]:
# Shift next entry time within the same commuter group
df3['next_ENTRY_TM'] = df3.groupby('CRD_NUM')['ENTRY_TM'].shift(-1)

# Time gap = next stage's entry time minus current stage's exit time (in mins)
df3['time_gap_mins'] = (
    df3['next_ENTRY_TM'] - df3['EXIT_TM']
).dt.total_seconds() / 60

# allowance (seconds) = walking distance (m) / walking speed (m/s), then convert to mins
# Rows with no walking distance → allowance is NaN
df3['transfer_allowance_mins'] = (
    df3['walk_distance'] / df3['walking_speed_ms']
) / 60

# True = gap exceeds allowance → new journey (not a transfer)
# If walk_distance is null → allowance is NaN → comparison returns NaN → fillna(True) marks as new journey
df3['exceeds_time_allowance'] = (
    (df3['time_gap_mins'] > df3['transfer_allowance_mins'])
).fillna(True)

print(df3['exceeds_time_allowance'].value_counts())
print(f"\nTotal temporal new journey flags: {df3['exceeds_time_allowance'].sum():,}")

print(f"Null walking distance: {df3['walk_distance'].isna().sum():,}")
print(f"Total rows: {len(df3):,}")
print(f"Null %: {df3['walk_distance'].isna().mean() * 100:.2f}%")

exceeds_time_allowance
True     2327847
False    1518937
Name: count, dtype: int64

Total temporal new journey flags: 2,327,847
Null walking distance: 1,239,514
Total rows: 3,846,784
Null %: 32.22%


#### 3. Spatial Criteria (Reasonable walking dist vs Actual walking dist)
- The spatial rule identifies whether the transition between consecutive journey stages is likely a transfer or the start of a new journey based on walking distance.
- It flags if the distance between the alighting location of the first journey stage and the boarding location of the next journey stage exceeds a reasonable transfer walking distance.
- Implementation: 
    - Reasonable transfer wwalking distance is the column, walking distance
    - Actual walking distance is time gap between tap out and the next tap in * walking speed
    - To account for time where commuter might be waiting for the next ride instead of walking, we include a 20% buffer.

In [15]:
# actual walking distance (m) = time gap (seconds) × walking speed (m/s)
# time_gap_mins already exists from temporal step, convert back to seconds
df3['actual_walking_dist_m'] = (
    df3['time_gap_mins'] * 60
) * df3['walking_speed_ms']

# Allowance = walk_distance (m) with 20% buffer
# If walk_distance is null → fillna(True) flags as new journey
df3['exceeds_walking_dist'] = (
    (df3['CRD_NUM'] == df3.groupby('CRD_NUM')['CRD_NUM'].shift(-1).reindex(df3.index)) |
    (df3['actual_walking_dist_m'] > df3['walk_distance'] * 1.2)
).fillna(True)

print(df3['exceeds_walking_dist'].value_counts())
print(f"\nTotal spatial new journey flags: {df3['exceeds_walking_dist'].sum():,}")

exceeds_walking_dist
True     2633794
False    1212990
Name: count, dtype: int64

Total spatial new journey flags: 2,633,794


#### 3. Spatial Criteria (Haversine):
- The spatial rule identifies whether the transition between consecutive journey stages is likely a transfer or the start of a new journey based on walking distance.
- Implementation: 
    - Use original clean df to access logtitude and latitude columns
    - Sort the dataset by card number (CRD_NUM) and timestamp (ENTRY_TM) to get consecutive stages for each rider.
    - For each ride, we look at the next stage’s boarding location (latitude and longitude). Rows without a next stage are ignored because there is nothing to compare. The person finished all their trips for the day. 
    - We create a boolean mask to select only rows that have a next stage (NEXT_ENTRY_LAT is not NaN). This helps us to ignore observations without next stages
    - Use Haversine formula to compute the shortest straight-line distance between DEST_LATITUDE & DEST_LONGTITUDE (alighting stop) and the NEXT_ENTRY_LONG & NEXT_ENTRY_LAT (the next entry point). Vectorisation is used because row-by-row computation was too slow. 
    - Allow a maximum transfer distance of 500 metres (we can change this)
    - If the distance exceeds a maximum allowed transfer distance (0.5 km), the row is flagged as spatial_new_journey = True. This indicates that the next ride is likely a new journey.

In [16]:
# Shift boarding coordinates to get "next stage" per rider
df3['NEXT_ENTRY_LAT'] = df3.groupby('CRD_NUM')['ORIG_LATITUDE'].shift(-1)
df3['NEXT_ENTRY_LON'] = df3.groupby('CRD_NUM')['ORIG_LONGITUDE'].shift(-1)

In [17]:
import numpy as np
#use vectorized function so it runs much faster than row by row
def haversine_vectorized(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    
    r = 6371  # Earth radius in km
    return c * r

In [18]:
#compute distance to next stage if it exists
mask = df3['NEXT_ENTRY_LAT'].notna()  # skip last stage (ie. if there is no next tap in) 
df3.loc[mask, 'distance_to_next_km'] = haversine_vectorized(
    df3.loc[mask, 'DEST_LATITUDE'].values,
    df3.loc[mask, 'DEST_LONGITUDE'].values,
    df3.loc[mask, 'NEXT_ENTRY_LAT'].values,
    df3.loc[mask, 'NEXT_ENTRY_LON'].values
)

In [19]:
#flag spatial transfers
# set max reasonable walking transfer distance = 0.5 km 
# we can change this threshold
max_transfer_distance_km = 0.5

#initialise as False. Only mark as True if distance to next stage exceeds threshold
df3['spatial_new_journey'] = False
df3.loc[mask, 'spatial_new_journey'] = df3.loc[mask, 'distance_to_next_km'] > max_transfer_distance_km

In [20]:
df3['spatial_new_journey'].value_counts()

spatial_new_journey
False    3657236
True      189548
Name: count, dtype: int64

# Internal Validity Check
- Drop rows with missing critical data
- Combine `pt_ride` and `pt_jrny` and only keep those where the ride entry time >= journey start time and ride exit time <= journey end time
- Generate confusion matrix calculate metrics (split rate, merge rate, sensitivity, specificity, accuracy)

In [ ]:
df5 = pd.read_pickle('../data/df5.pkl')

In [ ]:
df5.info()

In [ ]:
# Drop rows with missing critical data
critical_cols = [
    'JRNY_START_TM', 'JRNY_END_TM',
    'JRNY_ORIG_ID_NUM', 'JRNY_DEST_ID_NUM',
    'ORIG_LATITUDE', 'ORIG_LONGITUDE',
    'DEST_LATITUDE', 'DEST_LONGITUDE'
]

before = len(df5)
df5 = df5.dropna(subset=critical_cols).reset_index(drop=True)
after = len(df5)

print(f"Rows dropped: {before - after:,}")
print(f"Rows remaining: {after:,}")

In [ ]:
df4 = df5[df5['JRNY_START_DT'] == '2025-02-11']

#### 1. Binary Criteria

In [ ]:
# Combine pt_ride and pt_jrny and only keep those where the ride entry time = 
df6 = df2.merge(df4, on=['CRD_NUM', 'JRNY_ID_NUM'], suffixes=('', '_JRNY'))
df6 = df6[(df6['ENTRY_DT'] >= df6['JRNY_START_DT']) & (df6['EXIT_DT'] <= df6['JRNY_END_DT'])]

In [ ]:
# Shift needed columns
df6['next_JRNY_ID_NUM'] = df6['JRNY_ID_NUM'].shift(-1)

# Flag true single journeys
df6['true_same_journey'] = (df6['JRNY_ID_NUM'] == df6['next_JRNY_ID_NUM'])

print(df6['true_same_journey'].value_counts())

In [ ]:
# Confusion matrix
pd.crosstab(
    df6['true_same_journey'],
    df6['same_service_return'],
    rownames=['Actual'],
    colnames=['Predicted']
)

In [ ]:
total_true = (df6['true_same_journey'] == True).sum()

# Split Error Rate (FN)
split_error = ((df6['true_same_journey'] == True) & (df6['same_service_return'] == False)).sum()
split_rate = split_error / total_true
print("Split rate:", split_rate)

# Merge Error Rate (FP)
merge_error = ((df6['true_same_journey'] == False) & (df6['same_service_return'] == True)).sum()
merge_rate = merge_error / total_true
print("Merge rate:", merge_rate)

# Sensitivity (TP / (TP + FN))
tp = ((df6['true_same_journey'] == True) & (df6['same_service_return'] == True)).sum()
sensitivity = tp / (tp + split_error)
print("Sensitivity:", sensitivity)

# Specificity (TN / (TN + FP))
tn = ((df6['true_same_journey'] == False) & (df6['same_service_return'] == False)).sum()
specificity = tn / (tn + merge_error)
print("Specificity", specificity)

# Accuracy
accuracy = (tp + tn) / (total_true + merge_error + tn)
print("Accuracy:", accuracy)

#### 2. Temporal Criteria

In [ ]:
df6['true_new_journey'] = (df6['JRNY_ID_NUM'] != df6['next_JRNY_ID_NUM'])

In [ ]:
# Confusion matrix
pd.crosstab(
    df6['true_new_journey'],
    df6['temporal_new_journey'],
    rownames=['Actual'],
    colnames=['Predicted']
)

In [ ]:
# Split Error Rate (FN)
split_error = ((df6['true_new_journey'] == True) & (df6['temporal_new_journey'] == False)).sum()
split_rate = split_error / total_true
print("Split rate:", split_rate)

# Merge Error Rate (FP)
merge_error = ((df6['true_new_journey'] == False) & (df6['temporal_new_journey'] == True)).sum()
merge_rate = merge_error / total_true
print("Merge rate:", merge_rate)

# Sensitivity (TP / (TP + FN))
tp = ((df6['true_new_journey'] == True) & (df6['temporal_new_journey'] == True)).sum()
sensitivity = tp / (tp + split_error)
print("Sensitivity:", sensitivity)

# Specificity (TN / (TN + FP))
tn = ((df6['true_new_journey'] == False) & (df6['temporal_new_journey'] == False)).sum()
specificity = tn / (tn + merge_error)
print("Specificity", specificity)

# Accuracy
accuracy = (tp + tn) / (total_true + merge_error + tn)
print("Accuracy:", accuracy)

#### 3. Spatial Criteria

In [ ]:
# Combine pt_ride and pt_jrny and only keep those where the ride entry time = 
df7 = df3.merge(df5, on=['CRD_NUM', 'JRNY_ID_NUM'], suffixes=('', '_JRNY'))
df7 = df7[(df7['ENTRY_DT'] >= df7['JRNY_START_DT']) & (df7['EXIT_DT'] <= df7['JRNY_END_DT'])]

In [ ]:
# Shift needed columns
df7['next_JRNY_ID_NUM'] = df7['JRNY_ID_NUM'].shift(-1)

# Flag true single journeys
df7['true_same_journey'] = (df7['JRNY_ID_NUM'] == df7['next_JRNY_ID_NUM'])

print(df7['true_same_journey'].value_counts())

In [ ]:
# Confusion matrix
pd.crosstab(
    df7['true_new_journey'],
    df7['spatial_new_journey'],
    rownames=['Actual'],
    colnames=['Predicted']
)

In [ ]:
total_true2 = (df7['true_same_journey'] == True).sum()

# Split Error Rate (FN)
split_error = ((df7['true_new_journey'] == True) & (df7['spatial_new_journey'] == False)).sum()
split_rate = split_error / total_true2
print("Split rate:", split_rate)

# Merge Error Rate (FP)
merge_error = ((df7['true_new_journey'] == False) & (df7['spatial_new_journey'] == True)).sum()
merge_rate = merge_error / total_true2
print("Merge rate:", merge_rate)

# Sensitivity (TP / (TP + FN))
tp = ((df7['true_new_journey'] == True) & (df7['spatial_new_journey'] == True)).sum()
sensitivity = tp / (tp + split_error)
print("Sensitivity:", sensitivity)

# Specificity (TN / (TN + FP))
tn = ((df7['true_new_journey'] == False) & (df7['spatial_new_journey'] == False)).sum()
specificity = tn / (tn + merge_error)
print("Specificity", specificity)

# Accuracy
accuracy = (tp + tn) / (total_true2 + merge_error + tn)
print("Accuracy:", accuracy)